# 02 — Data Cleaning and Preparation

## tl;dr

This notebook applies documented, non-destructive cleaning; constructs scope-aware holiday and daily oil features; validates merges; and selects a manageable 24-month analytical subset using observed sales volume and variability.

## Context & Methods

Raw files remain unchanged. Outliers and zero sales are flagged and profiled rather than removed. Store, transaction, oil, and holiday data are attached with explicit join validation. The full panel is retained in memory for high-level checks; only the justified small subset is saved to avoid creating another large copy.

### Key assumptions

- Non-transferred national holidays apply to all stores; regional holidays match state; local holidays match city.
- Missing oil values are forward-filled on a daily calendar, with a start-boundary backfill only if needed; imputation is flagged.
- Missing transaction records are retained as missing, not converted to zero.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
candidate = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (candidate, *candidate.parents)
    if (path / "data").is_dir() and (path / "src").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DATA_DIR = PROJECT_ROOT / "outputs" / "processed_data"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
for directory in (PROCESSED_DATA_DIR, OUTPUT_DATA_DIR, FIGURE_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.data_loader import load_all_available
from src.preprocessing import build_analysis_panel, choose_analysis_scope

pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
datasets = load_all_available(RAW_DATA_DIR)
train = datasets["train.csv"]

## Data

### 1. Systematic source-quality checks

In [ ]:
quality_checks = []
for filename, frame in datasets.items():
    quality_checks.append(
        {
            "file": filename,
            "rows": len(frame),
            "exact_duplicates": int(frame.duplicated().sum()),
            "missing_cells": int(frame.isna().sum().sum()),
            "invalid_dates": int(frame["date"].isna().sum()) if "date" in frame else 0,
        }
    )
display(pd.DataFrame(quality_checks))

print("Negative sales:", int(train["sales"].lt(0).sum()))
print("Zero sales:", f"{train['sales'].eq(0).sum():,}", f"({100*train['sales'].eq(0).mean():.2f}%)")
print("Negative promotions:", int(train["onpromotion"].lt(0).sum()))
print("Maximum onpromotion:", int(train["onpromotion"].max()))

### 2. Investigate extremes without deleting them

In [ ]:
sales_quantiles = train["sales"].quantile([0, .5, .9, .95, .99, .999, 1]).rename("sales")
promotion_quantiles = train["onpromotion"].quantile([0, .5, .9, .95, .99, .999, 1]).rename("onpromotion")
display(pd.concat([sales_quantiles, promotion_quantiles], axis=1))

extreme_threshold = train["sales"].quantile(.999)
extreme_rows = train.loc[train["sales"].ge(extreme_threshold)].copy()
extreme_context = (
    extreme_rows.groupby("family")["sales"]
    .agg(extreme_rows="size", median_extreme_sales="median", max_sales="max")
    .sort_values("extreme_rows", ascending=False)
    .head(10)
)
display(extreme_context)
print("Decision: retain extremes; later notebooks test holiday, promotion, timing, and family context.")

### 3. Inspect oil, holidays, and transactions before integration

In [ ]:
oil = datasets["oil.csv"]
holidays = datasets["holidays_events.csv"]
transactions = datasets["transactions.csv"]

context_quality = pd.DataFrame(
    [
        ("oil missing price", int(oil["dcoilwtico"].isna().sum()), 100 * oil["dcoilwtico"].isna().mean()),
        ("holiday exact duplicates", int(holidays.duplicated().sum()), 100 * holidays.duplicated().mean()),
        ("holiday extra rows on repeated dates", int(holidays.duplicated("date").sum()), 100 * holidays.duplicated("date").mean()),
        ("transaction exact duplicates", int(transactions.duplicated().sum()), 100 * transactions.duplicated().mean()),
        ("transaction negative counts", int(transactions["transactions"].lt(0).sum()), 100 * transactions["transactions"].lt(0).mean()),
    ],
    columns=["check", "count", "pct_of_source_rows"],
)
display(context_quality)
display(holidays.groupby(["locale", "type", "transferred"]).size().rename("records").reset_index())

## Results

### 4. Build and validate the integrated panel

In [ ]:
panel = build_analysis_panel(datasets)
print(f"Rows before merge: {len(train):,}")
print(f"Rows after merge:  {len(panel):,}")
print("Unique date-store-family key:", not panel.duplicated(["date", "store_nbr", "family"]).any())
print("Missing store attributes:", int(panel[["city", "state", "type", "cluster"]].isna().sum().sum()))
print("Missing oil after daily fill:", int(panel["dcoilwtico"].isna().sum()))
print("Sales rows whose store-date lacks transactions:", f"{panel['transaction_record_missing'].sum():,}")
display(panel.head())

### 5. Explain missing transaction coverage

In [ ]:
store_date = panel.groupby(["store_nbr", "date"], observed=True).agg(
    sales=("sales", "sum"),
    transactions=("transactions", "max"),
).reset_index()
missing_tx = store_date.loc[store_date["transactions"].isna()]
tx_by_store = store_date.groupby("store_nbr").agg(
    observed_store_dates=("date", "size"),
    transaction_coverage_pct=("transactions", lambda values: 100 * values.notna().mean()),
    first_positive_sales_date=("date", "min"),
).sort_values("transaction_coverage_pct")
display(tx_by_store.head(10))
print(f"Missing transaction store-dates: {len(missing_tx):,} of {len(store_date):,} ({100*len(missing_tx)/len(store_date):.2f}%)")
print("Decision: retain missingness; do not interpret missing transaction rows as zero traffic.")

### 6. Select the detailed-analysis scope from observed evidence

In [ ]:
selected_stores, selected_families, scope_start = choose_analysis_scope(panel)
scope_end = panel["date"].max()

store_daily = panel.groupby(["store_nbr", "date"], observed=True)["sales"].sum().reset_index()
store_metrics = store_daily.groupby("store_nbr")["sales"].agg(total_sales="sum", mean_daily_sales="mean", std_daily_sales="std")
store_metrics["cv"] = store_metrics["std_daily_sales"] / store_metrics["mean_daily_sales"]

family_daily = panel.groupby(["family", "date"], observed=True)["sales"].sum().reset_index()
family_metrics = family_daily.groupby("family", observed=True)["sales"].agg(total_sales="sum", mean_daily_sales="mean", std_daily_sales="std")
family_metrics["cv"] = family_metrics["std_daily_sales"] / family_metrics["mean_daily_sales"]

display(store_metrics.loc[selected_stores].sort_values("total_sales", ascending=False))
display(family_metrics.loc[selected_families].sort_values("total_sales", ascending=False))
print("Selected stores:", selected_stores)
print("Selected families:", selected_families)
print("Detailed period:", scope_start.date(), "to", scope_end.date())

### 7. Save only the manageable analysis subset and scope metadata

In [ ]:
subset = panel.loc[
    panel["store_nbr"].isin(selected_stores)
    & panel["family"].isin(selected_families)
    & panel["date"].between(scope_start, scope_end)
].copy()

subset_path = OUTPUT_DATA_DIR / "analysis_subset.csv.gz"
subset.to_csv(subset_path, index=False, compression="gzip")
scope = {
    "selected_stores": selected_stores,
    "selected_families": selected_families,
    "start_date": scope_start.strftime("%Y-%m-%d"),
    "end_date": scope_end.strftime("%Y-%m-%d"),
    "selection_logic": (
        "Stores: two highest-volume, one median-volume, one lowest-volume, and one remaining high-CV store. "
        "Families: four highest-volume plus the highest-CV family among the top 15 by volume."
    ),
}
(OUTPUT_DATA_DIR / "analysis_scope.json").write_text(json.dumps(scope, indent=2))
print(f"Saved {len(subset):,} rows to {subset_path.relative_to(PROJECT_ROOT)}")
print(f"Subset size: {subset_path.stat().st_size / 1024**2:.2f} MiB compressed")

## Takeaways

In [ ]:
display(Markdown(f'''
- The cleaned integration preserves all **{len(panel):,}** training rows and the intended date-store-family key.
- No negative sales or promotion values were found; no rows were removed as outliers.
- **{100*panel['transaction_record_missing'].mean():.2f}% of sales rows** inherit a missing transaction record at their store-date. This is retained as missing and flagged.
- Oil is aligned to a daily calendar and filled from prior information, with an explicit imputation flag.
- Holidays are matched to stores by national, regional, and local scope, avoiding a many-to-many date-only join.
- Detailed analysis uses **{len(selected_stores)} stores**, **{len(selected_families)} families**, and the final **24 months**, selected through transparent volume and variability criteria.
- The full panel is rebuilt from raw inputs when needed; only the **{len(subset):,}-row** subset is saved, avoiding a large duplicate of the raw sales data.
'''))